# Day 010：`eval_llm.py` 外层调用链

本 Notebook 与 [Day 010 互动档案](../day-010.md) 配套。它沿着真实执行入口运行可以独立观察的部分：命令行参数、模型配置、tokenizer、chat template、`BatchEncoding`、生成控制参数、结果切片、对话历史和速度统计。

不要把 Cell 运行成功当成已经理解。每个小节先阅读说明，再执行代码，用自己的话解释输出。完整模型生成放在最后的可选 Cell；今天的源码阅读停在 `model.generate(...)` 的外层，不提前展开其内部循环。

## 1. 今天沿哪条调用链阅读

```text
eval_llm.py 直接执行
-> main()
-> argparse 读取命令行
-> init_model(args)
-> conversation / chat template
-> tokenizer
-> model.generate(...)
-> decode 新生成部分
-> 保存 assistant 回复并统计速度
```

今天不从 Attention 中间函数跳读；下一学习日再从 `generate()` 内部继续。

In [ ]:
from pathlib import Path

# 兼容从仓库根目录或 Notebook 目录启动 Jupyter。
candidates = [Path.cwd(), Path.cwd().parent.parent, Path('/home/zcf/githubs/minimind')]
repo_root = next(path for path in candidates if (path / 'minimind' / 'eval_llm.py').exists())
script_path = repo_root / 'minimind' / 'eval_llm.py'
model_path = repo_root / 'minimind' / 'minimind-3'
print('仓库根目录：', repo_root)
print('入口脚本：', script_path)
print('导出模型目录存在：', model_path.exists())

## 2. 入口保护和实际源码

脚本末尾是：

```python
if __name__ == "__main__":
    main()
```

直接执行 `python eval_llm.py` 时才会调用 `main()`；被别的文件导入时不会自动启动交互程序。下面只读取源码，不执行 `main()`。

In [ ]:
source_lines = script_path.read_text(encoding='utf-8').splitlines()
for line_number, line in list(enumerate(source_lines[:3], start=1)) + list(enumerate(source_lines[-4:], start=len(source_lines) - 3)):
    print(f'{line_number:>3}: {line}')

## 3. 参数解析：命令行文本变成 `args`

真实运行命令是：

```bash
python eval_llm.py --load_from ./minimind-3 --device cpu --max_new_tokens 64 --temperature 0.1 --top_p 0.9 --show_speed 1
```

`input()` 得到的命令行参数最初都是文本；`type=int` 或 `type=float` 会按规则转换。下面用一个不启动模型的最小 parser 重现本次关键值。

In [ ]:
import argparse

parser = argparse.ArgumentParser()
parser.add_argument('--load_from', default='model', type=str)
parser.add_argument('--device', default='cpu', type=str)
parser.add_argument('--max_new_tokens', default=8192, type=int)
parser.add_argument('--temperature', default=0.85, type=float)
parser.add_argument('--top_p', default=0.95, type=float)
parser.add_argument('--show_speed', default=1, type=int)
args = parser.parse_args([
    '--load_from', './minimind-3', '--device', 'cpu',
    '--max_new_tokens', '64', '--temperature', '0.1',
    '--top_p', '0.9', '--show_speed', '1'
])
print(vars(args))
print('max_new_tokens 类型：', type(args.max_new_tokens).__name__)
print('temperature 类型：', type(args.temperature).__name__)

## 4. `config.json` 变成具体配置对象

`AutoConfig` 读取结构配置，告诉自动模型工厂应该创建哪一种具体模型。它只提供结构信息，不包含训练后的权重数值。

In [ ]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained(str(model_path))
print('配置对象类型：', type(config).__name__)
print('model_type：', config.model_type)
print('architectures：', config.architectures)
print('hidden_size：', config.hidden_size)
print('num_hidden_layers：', config.num_hidden_layers)

本地 `./minimind-3` 的实际关系是：

```text
config.json -> Qwen3Config -> Qwen3ForCausalLM 的结构
model.safetensors -> 训练后的参数数值
```

`AutoModelForCausalLM.from_pretrained(...)` 会按配置选择具体模型类，再加载权重；`AutoModelForCausalLM` 和 `AutoConfig` 是自动选择入口，最终对象是具体的 `Qwen3ForCausalLM` 和 `Qwen3Config`。

## 5. 消息字典经过 chat template

`conversation` 保存结构化消息，`apply_chat_template(..., tokenize=False, add_generation_prompt=True)` 只负责把它排成模型协议字符串，还没有产生 token ID。

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(str(model_path))
conversation = [{'role': 'user', 'content': '做一下自我介绍'}]
formatted_text = tokenizer.apply_chat_template(
    conversation,
    tokenize=False,
    add_generation_prompt=True,
    open_thinking=False,
)
print(formatted_text)
print('formatted_text 类型：', type(formatted_text).__name__)

末尾的 assistant 起始标记表示：接下来需要模型续写 assistant 回合。`tokenize=False` 很重要，它让这一格仍然返回字符串；下一格才真正调用 tokenizer。

## 6. 字符串变成 `BatchEncoding`

```python
model_inputs = tokenizer(
    formatted_text,
    return_tensors='pt',
    truncation=True,
).to('cpu')
```

`return_tensors='pt'` 让结果使用 PyTorch Tensor；`truncation=True` 允许过长输入被截断；`.to('cpu')` 移动其中的 Tensor，与 CPU 上的模型保持设备一致。

In [ ]:
model_inputs = tokenizer(
    formatted_text,
    return_tensors='pt',
    truncation=True,
).to('cpu')
print('model_inputs 类型：', type(model_inputs).__name__)
print('input_ids shape：', model_inputs['input_ids'].shape)
print('attention_mask shape：', model_inputs['attention_mask'].shape)
print('input_ids device：', model_inputs['input_ids'].device)
print('attention_mask：', model_inputs['attention_mask'])
print('整体解码：', tokenizer.decode(model_inputs['input_ids'][0]))

预期会看到一条输入、21 个 token，即 shape `[1, 21]`。本次没有 padding，所以 `attention_mask` 全为 1；`<|im_start|>`、`<|im_end|>`、`<think>` 等协议 token 虽然是特殊 token，但属于模型需要读取的有效输入。

## 7. 生成参数的职责边界

本次真实调用中的关键参数：

| 参数 | 本次值 | 作用 |
| --- | ---: | --- |
| `max_new_tokens` | 64 | 最多新增 64 个 token，不含输入 |
| `eos_token_id` | 2 | 生成结束标记后可以提前停止 |
| `pad_token_id` | 0 | 序列补齐和结束序列维护 |
| `do_sample` | `True` | 按概率采样，而非总选最高分 |
| `temperature` | 0.1 | 让概率更集中，更偏向高分 token |
| `top_p` | 0.9 | 保留累计概率达到 0.9 的高概率候选 |
| `repetition_penalty` | 1 | 不惩罚已出现的 token |
| `streamer` | `TextStreamer` | 边生成边解码显示 |

输入长度为 21 时，最多总长度是 `21 + 64 = 85`；如果提前生成 EOS，实际长度会更短。

### `top_p` 小实验

候选概率按从高到低排列：`A=0.50`、`B=0.30`、`C=0.15`、`D=0.05`。累计到 `A+B` 只有 `0.80`，加上 `C` 后是 `0.95`，因此 `top_p=0.9` 会保留 `A、B、C`，排除 `D`。保留数量不是固定值，而由当前概率分布决定。

In [ ]:
import torch

candidate_names = ['A', 'B', 'C', 'D']
candidate_probs = torch.tensor([0.50, 0.30, 0.15, 0.05])
cumulative = candidate_probs.cumsum(dim=0)
keep = cumulative <= 0.9
# top-p 还必须保留第一个使累计概率达到阈值的候选。
first_reaching = int(torch.nonzero(cumulative >= 0.9, as_tuple=False)[0].item())
keep[:first_reaching + 1] = True
print('候选：', candidate_names)
print('概率：', candidate_probs.tolist())
print('累计概率：', cumulative.tolist())
print('保留：', [name for name, flag in zip(candidate_names, keep) if flag])
print('排除：', [name for name, flag in zip(candidate_names, keep) if not flag])

## 8. `generated_ids` 为什么要切片

`generate()` 返回的序列通常包含：`原输入 token IDs + 新生成 token IDs`。因此必须从输入长度位置开始切片，再 decode；否则 `response` 会把原来的提问和模板也包含进去。

In [ ]:
input_length = model_inputs['input_ids'].shape[1]
# 这里用一个教学用的假设序列表示“输入 + 3 个新 token”。
new_token_ids = torch.tensor([[463, 3593, tokenizer.eos_token_id]])
generated_ids = torch.cat([model_inputs['input_ids'], new_token_ids], dim=1)
new_ids = generated_ids[0][input_length:]
response = tokenizer.decode(new_ids, skip_special_tokens=True)
print('输入 token 数：', input_length)
print('完整序列 token 数：', generated_ids.shape[1])
print('新生成 token 数：', len(new_ids))
print('切片后的 ID：', new_ids.tolist())
print('decode 后 response：', repr(response))
print('完整序列直接 decode：', repr(tokenizer.decode(generated_ids[0])))

这里的 `generated_ids[0]` 取 batch 中第一个样本；`generated_ids[0][input_length:]` 只留下新生成部分。EOS 被 `skip_special_tokens=True` 隐藏，所以本例的 `response` 只显示普通文本。

## 9. assistant 历史与 `historys`

模型回答解码后，源码执行：

```python
conversation.append({'role': 'assistant', 'content': response})
```

下一轮开始时，`historys=0` 会清空列表；`historys=2` 会保留上一轮的 user/assistant 两条消息，再追加新的 user 消息。

In [ ]:
conversation = [{'role': 'user', 'content': '做一下自我介绍'}]
response = '我是一个小参数语言模型。'
conversation.append({'role': 'assistant', 'content': response})
print('追加 assistant 后：', conversation)
print('historys=0 的下一轮：', conversation[-0:] if False else [])
print('historys=2 的下一轮：', conversation[-2:])

## 10. 新生成 token 数和速度

源码公式：

```python
gen_tokens = len(generated_ids[0]) - len(inputs['input_ids'][0])
speed = gen_tokens / (time.time() - st)
```

它统计 token 数而不是字符数；速度单位是 `tokens/second`。例如生成 20 个 token、耗时 4 秒，速度就是 5 tokens/s。

In [ ]:
generated_token_count = generated_ids.shape[1] - input_length
elapsed_seconds = 4.0
print('本教学序列的新生成 token 数：', generated_token_count)
print('假设生成 20 个 token、耗时 4 秒：', 20 / elapsed_seconds, 'tokens/s')

## 11. 可选：真正调用一次模型

这一格会加载约 64M 参数的本地模型并执行真实生成；CPU 上可能需要等待。它用于复现 `eval_llm.py` 外层行为，不要求今天阅读 `generate()` 内部实现。运行前确认当前 Python 环境已经安装仓库依赖。

In [ ]:
# 如需实际运行，取消下面代码的注释。
# import time
# import torch
# from transformers import AutoModelForCausalLM, TextStreamer
# model = AutoModelForCausalLM.from_pretrained(str(model_path), trust_remote_code=True)
# model = model.half().eval().to('cpu')
# streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
# st = time.time()
# generated_ids = model.generate(
#     inputs=model_inputs['input_ids'],
#     attention_mask=model_inputs['attention_mask'],
#     max_new_tokens=64, do_sample=True, streamer=streamer,
#     pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
#     top_p=0.9, temperature=0.1, repetition_penalty=1,
# )
# response = tokenizer.decode(generated_ids[0][input_length:], skip_special_tokens=True)
# print('response：', response)
# print('gen_tokens：', len(generated_ids[0]) - input_length)
# print('tokens/s：', (len(generated_ids[0]) - input_length) / (time.time() - st))

## Day 010 自检

完成 Notebook 后，应能用自己的话回答：

- 为什么 `./minimind-3` 会走 `Qwen3ForCausalLM`？
- `formatted_text` 和 `model_inputs` 的类型、职责有什么不同？
- 为什么 `input_ids` 和 `attention_mask` 的 shape 都是 `[1, 21]`？
- `max_new_tokens=64` 为什么不是总长度 64？
- `streamer` 和 `generated_ids` 为什么可以同时存在？
- 为什么 decode 前要从输入长度处切片？
- `historys=0` 和 `historys=2` 对下一轮输入有什么不同？

下一学习日从 `model.generate(...)` 内部开始：先看它如何初始化逐 token 生成，再进入 `forward`、最后位置 logits 和 KV Cache。